## Using a local LLM

Running a local LLM is possible usign ollama.  This has the advantages of not having to pay for an external service and now sharing your data with an external company.  The disadvantages are that you need to have a PC capable of hostign the LLM and the size of model you host is limited to the memory on a consumer grade GPU.  This means a local LLM will typically be less powerful than the LLM's provided by one of the major providers.

The OpenAI provided tool, WebSearchTool, does not appear to be compatible with running a local LLM, so this code sample does not use it. For now, the agent will operate without it, only using the server-filesystem MCP server to save files to disk.

It appears that the OpenAI traces functionality still works, so it can be used to introspect what the Agent is doing.

In [1]:
import os
from dotenv import load_dotenv, find_dotenv, dotenv_values
from openai import AsyncOpenAI
from agents import Agent, Runner, Tool, WebSearchTool, trace, function_tool, OpenAIChatCompletionsModel
from agents.mcp import MCPServerStdio
from instructions import TripPlannerInstructions

# Locate .env in this directory or any parent directory
dotenv_path = find_dotenv()
if not dotenv_path:
    raise FileNotFoundError('.env not found in repository or parent directories')

# Load into os.environ (preserves existing variables unless overridden by .env)
load_dotenv(dotenv_path, override=False)
# Also read raw values as a dict (useful to expose into notebook globals)
env = {k: v for k, v in dotenv_values(dotenv_path).items() if v is not None}

# Export into notebook globals for easy access by name
globals().update(env)

print('Loaded .env from', dotenv_path)
print('Loaded keys:', list(env.keys()))

Loaded .env from /media/nathan/linux_ssd/github/agentic_ai_trip_planner/.env
Loaded keys: ['OPENAI_API_KEY', 'GROQ_API_KEY', 'PUSHOVER_USER', 'PUSHOVER_TOKEN', 'SENDGRID_API_KEY', 'GOOGLE_API_KEY', 'SERPER_API_KEY', 'LANGSMITH_TRACING', 'LANGSMITH_ENDPOINT', 'LANGSMITH_API_KEY', 'LANGSMITH_PROJECT', 'POLYGON_API_KEY', 'POLYGON_PLAN', 'BRAVE_API_KEY']


## Using ollama 

The code here is similar to the previous notebooks, just updated to utilize ollama as the LLM provider. 

In [ ]:

# Generate custom instructions for the trip planner agent
planner = TripPlannerInstructions(
    output_file="trip_plan_using_ollama.md"
)
custom_instructions = planner.get_instructions()
print(custom_instructions)


# Set the base_url to your local Ollama instance
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# Set a dummy API key (required by the SDK, but not used by Ollama)
DUMMY_API_KEY = "ollama"

# Initialize the AsyncOpenAI client with the custom base_url
client = AsyncOpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=DUMMY_API_KEY,
)

# Specify the model you pulled with Ollama
# Note: Ollama expects just the model name (e.g., "llama3"), 
# not the full "gpt-oss" naming convention from the OpenAI API
OLLAMA_MODEL_NAME = "gpt-oss:20b" 

# Wrap the client in the Agents SDK model class
model = OpenAIChatCompletionsModel(
    openai_client=client,
    model=OLLAMA_MODEL_NAME
)


sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "output"))
filesystem_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}

#web_search_tool = WebSearchTool(search_context_size="low")

async with MCPServerStdio(params=filesystem_params, client_session_timeout_seconds=30) as mcp_server_files:
    trip_planner_agent = Agent(
        model=model,
        name="Trip Planner Agent",
        instructions="An agent that helps users plan trips by searching for destinations, accommodations, and activities.",
        #tools=[web_search_tool],  # This tool is not compatible with Ollama at this time.
        mcp_servers=[mcp_server_files]
    )
    with trace("Trip Planner Agent Ollama"):
        result = await Runner.run(trip_planner_agent, custom_instructions)
        print(result.final_output)

You are a methodical and detail-oriented trip planning assistant. Your task is to create a COMPLETE, timeline-based trip itinerary with specific departure/arrival times and durations for every activity.

CRITICAL: You must complete the ENTIRE itinerary before finishing. Do not stop at research phase. Do not ask for permission to continue. Work through all steps until you have a fully detailed day-by-day schedule.

The customer has provided the following details for their trip:
- Home Location: Columbus, OH
- Departure Date: 2026/02/25
- Return Date: 2026/03/07
- Destination: Tokyo, Japan
- Must-Do Activities: Visit the Tokyo Tower, Explore Akihabara, Experience a traditional tea ceremony, Visit the Tsukiji Fish Market, Take a day trip to Mount Fuji   
- Number of Travelers: 3
- Ages of Travelers: 51, 50, 17
- Other Considerations: 
  - I will be running in the Tokyo marathon on Sunday March 1, so I only need a relaxing place to eat on that day.
  - Starting on March 3, throughout the r